In [1]:
import sys; sys.path.append("..")
from agents.stats_agent import ask_stats_agent
from judges.deterministic import verify_numbers
from data.loader import get_player_stats

[07/10/26 23:07:40] INFO     No custom team name replacements found. You can configure these in       ]8;id=5707075;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=5707076;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/jade/soccerdata/config/teamname_replacements.json.                             

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=5707082;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=5707083;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_config.py#190\190]8;;\
                             /Users/jade/soccerdata/config/league_dict.json.                                       

                    INFO     Saving cached data to /Users/jade/soccerdata/data/FBref                 ]8;id=5707090;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=5707091;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/soccerdata/_common.py#250\250]8;;\

In [7]:
out = ask_stats_agent("How many goals did Saka score this season?")
print(out["answer"])
print(len(out["tool_results"]))   # 1 기대

[07/10/26 23:07:03] INFO     HTTP Request: POST                                                     ]8;id=8185881;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8185882;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 503 Service Unavailable"                                 

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [2]:
def scout_pipeline(question):
    out = ask_stats_agent(question)

    # No tool used (e.g. "who is the best player") -> nothing to verify
    if not out["tool_results"]:
        return {"answer": out["answer"], "verdict": "not_applicable"}

    # Verify the answer against every tool result it was based on
    verdicts = [verify_numbers(out["answer"], src) for src in out["tool_results"]]
    overall = "pass" if all(v["verdict"] == "pass" for v in verdicts) else "fail"

    return {"answer": out["answer"], "verdict": overall, "details": verdicts}

In [3]:
def scout_with_retry(question, max_attempts=2):
    for attempt in range(max_attempts):
        result = scout_pipeline(question)

        if result["verdict"] in ("pass", "not_applicable"):
            result["attempts"] = attempt + 1
            return result

        # Failed: tell the agent what could not be verified and retry
        bad = [n for v in result["details"] for n in v["unverified"]]
        question = (f"{question}\n\nYour previous answer contained numbers "
                    f"that could not be verified against the database: {bad}. "
                    f"Answer again using ONLY numbers from the tool results.")

    result["attempts"] = max_attempts
    result["warning"] = "still failing after retries"
    return result

In [4]:
# pass 기대
print(scout_pipeline("How many goals did Saka score this season?"))

# not_applicable 기대
print(scout_pipeline("Who is the best player in the world?"))

[07/10/26 23:08:02] INFO     HTTP Request: POST                                                     ]8;id=5707098;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707099;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'season': '2425', 'name': 'Saka'})


[07/10/26 23:08:03] INFO     HTTP Request: POST                                                     ]8;id=5707104;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707105;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

{'answer': 'Bukayo Saka scored 6 goals for Arsenal in the 2024-25 Premier League season.', 'verdict': 'pass', 'details': [{'verdict': 'pass', 'claimed': [6.0], 'unverified': []}]}


                    INFO     HTTP Request: POST                                                     ]8;id=5707110;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707111;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

{'answer': 'I can only provide player statistics. I cannot determine who the "best" player is.', 'verdict': 'not_applicable'}


In [5]:
print(scout_with_retry("How many total goals did Rashford score across both his clubs?"))

[07/10/26 23:09:35] INFO     HTTP Request: POST                                                     ]8;id=5707116;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707117;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Rashford', 'season': '2425'})


[07/10/26 23:09:36] INFO     HTTP Request: POST                                                     ]8;id=5707122;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707123;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

{'answer': 'In the 2024-25 Premier League season, Marcus Rashford scored 2 goals for Aston Villa and 4 goals for Manchester Utd.', 'verdict': 'pass', 'details': [{'verdict': 'pass', 'claimed': [2.0, 4.0], 'unverified': []}], 'attempts': 1}


In [6]:
print(scout_with_retry("What is Rashford's combined goal total for the season? Give me a single number, add both clubs together."))

[07/10/26 23:24:54] INFO     HTTP Request: POST                                                     ]8;id=5707128;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707129;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'season': '2425', 'name': 'Marcus Rashford'})


[07/10/26 23:24:56] INFO     HTTP Request: POST                                                     ]8;id=5707134;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707135;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

{'answer': "Marcus Rashford scored 2 goals for Aston Villa in the 2024-25 Premier League season and 4 goals for Manchester Utd in the 2024-25 Premier League season. I cannot provide a combined goal total, as per my instructions to report each team's numbers separately.", 'verdict': 'pass', 'details': [{'verdict': 'pass', 'claimed': [2.0, 4.0], 'unverified': []}], 'attempts': 1}


In [7]:
# Temporarily replace the pipeline with a fake that fails once, then passes
calls = {"n": 0}

def fake_pipeline(question):
    calls["n"] += 1
    if calls["n"] == 1:
        return {"answer": "Rashford scored 6 goals in total.",
                "verdict": "fail",
                "details": [{"verdict": "fail", "claimed": [6.0], "unverified": [6.0]}]}
    return {"answer": "4 goals for Manchester Utd, 2 for Aston Villa.",
            "verdict": "pass", "details": []}

real_pipeline = scout_pipeline      # keep the real one
scout_pipeline = fake_pipeline      # swap in the fake
print(scout_with_retry("test"))     # attempts: 2 가 나와야 루프 정상
scout_pipeline = real_pipeline      # swap back

{'answer': '4 goals for Manchester Utd, 2 for Aston Villa.', 'verdict': 'pass', 'details': [], 'attempts': 2}


In [8]:
from orchestrator.pipeline import scout_with_retry as pipeline_from_module
print(pipeline_from_module("How many goals did Saka score this season?"))

[07/10/26 23:27:56] INFO     HTTP Request: POST                                                     ]8;id=5707140;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707141;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

  [tool call] get_player_stats({'name': 'Saka', 'season': '2425'})


[07/10/26 23:27:57] INFO     HTTP Request: POST                                                     ]8;id=5707146;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=5707147;file:///Users/jade/Project/AI-Football-Scout/venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

{'answer': 'Bukayo Saka scored 6 goals for Arsenal in the 2024-25 Premier League season.', 'verdict': 'pass', 'details': [{'verdict': 'pass', 'claimed': [6.0], 'unverified': []}], 'attempts': 1}
